# AIDA 1141 Final Project – 01 Labels and Split

**Data version 2** (`shared/synthetic-data/`). This notebook defines which assessments can be used for modelling, confirms that every model input is available at the assessment time, creates the fixed employee-level split, and defines a time-based check before any model is trained.

The question for the final project: can a model flag onboarding tasks at risk of missing the employee's start date better than simple follow-up rules, at a review volume the coordinator could manage? All records are synthetic; counts below are observed dataset counts, not evidence about real employees.

In [1]:
import importlib.util
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
SEED = 2026

read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
requests = read(SD / "cleaning/output/requests_cleaned.csv")            # official cleaned requests
assessments = read(SD / "requests/data/assessments.csv")                # accurate by construction
events = read(SD / "requests/data/status_events.csv")                   # accurate by construction
cleaning_info = json.loads((SD / "cleaning/output/requests_cleaned_info.json").read_text())

# Business-day rules (closure dates) and the extract time, taken from the data version 2 generator
spec = importlib.util.spec_from_file_location("generator", SD / "requests/generate_requests.py")
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)
EXTRACT_AT = pd.Timestamp(generator.EXTRACT_AT)

for frame, cols in [(requests, ["submitted_at", "info_complete_at", "assigned_at", "team_completed_at"]),
                    (assessments, ["assessed_at"]), (events, ["event_at"])]:
    for c in cols:
        frame[c] = pd.to_datetime(frame[c].replace("", None), format="%Y-%m-%dT%H:%M")
requests["confirmed_start_date"] = pd.to_datetime(requests["confirmed_start_date"].replace("", None))

print(f"Data version {DATA_VERSION}; cleaning version {cleaning_info['cleaning_version']}; extract {EXTRACT_AT:%Y-%m-%d %H:%M}")
print(f"{len(requests):,} cleaned requests, {len(assessments):,} assessments, {len(events):,} status events")

Data version 2; cleaning version 1.0; extract 2026-09-29 17:00
4,364 cleaned requests, 5,413 assessments, 18,491 status events


## A. Data and label checks

Each assessment is joined to its cleaned request. A task is **at risk** if the responsible team finishes after 08:00 on the confirmed start date, in a simulation with no coordinator follow-up.

At the data-extract date (17:00, 29 September 2026), an outcome is **known** if the team had finished, or if the task was unfinished but its deadline had already passed (a known missed deadline). It is **unknown** only when the task was unfinished and its deadline was still ahead. A blank completion time alone does not make an outcome unknown.

Rather than trusting the stored labels, the checks recalculate every label and the overdue flag independently from the completion time, the deadline, the assessment time, and the extract date. They also confirm unique IDs, that each label column contains only its documented values, one label per task across its checkpoints, and no assessments after the extract.

In [2]:
data = assessments.merge(
    requests[["request_id", "employee_id", "task_type", "responsible_team", "hiring_department", "employee_type",
              "intake_channel", "info_issue_at_intake", "info_complete_at", "assigned_at", "team_completed_at",
              "confirmed_start_date", "ready_by_start"]].rename(columns={"employee_id": "request_employee_id"}),
    on="request_id", how="left", validate="many_to_one")
data["label_known"] = data["at_risk"] != ""
data["deadline"] = data["confirmed_start_date"] + pd.Timedelta(hours=8)

# Labels recalculated independently from the timestamps, the deadline, and the extract time
finished = data["team_completed_at"].notna()
expected_label = np.select(
    [finished, data["deadline"] <= EXTRACT_AT],
    [np.where(data["team_completed_at"] > data["deadline"], "True", "False"), "True"],
    default="")                      # unfinished with a future (or unconfirmed) deadline: unknown
expected_overdue = np.where(data["assessed_at"] >= data["deadline"], "True", "False")

checks = pd.Series({
    "Cleaned request IDs are unique": requests["request_id"].is_unique,
    "Assessment IDs are unique": data["assessment_id"].is_unique,
    "Every assessment matches one cleaned request": data["request_id"].isin(requests["request_id"]).all(),
    "Assessment and request employee IDs agree": (data["employee_id"] == data["request_employee_id"]).all(),
    "at_risk is True, False, or blank only": data["at_risk"].isin(["True", "False", ""]).all(),
    "ready_by_start is True, False, or blank only": data["ready_by_start"].isin(["True", "False", ""]).all(),
    "past_deadline_at_assessment is True or False only": data["past_deadline_at_assessment"].isin(["True", "False"]).all(),
    "Each task has one label across its checkpoints": data.groupby("request_id")["at_risk"].nunique().max() == 1,
    "No assessment after the extract": (data["assessed_at"] <= EXTRACT_AT).all(),
    "Labels recalculate from timestamps (completed: finished after the deadline; unfinished: deadline passed = at risk, future deadline = unknown)":
        (data["at_risk"] == expected_label).all(),
    "Overdue flag recalculates as assessed at or after the deadline":
        (data["past_deadline_at_assessment"] == expected_overdue).all(),
    "Known overdue assessments are always at risk":
        (data.loc[data["label_known"] & (data["past_deadline_at_assessment"] == "True"), "at_risk"] == "True").all(),
    "Assessment labels agree with the requests' readiness (opposite values; both blank when unknown)":
        ((data["at_risk"] == "") == (data["ready_by_start"] == "")).all()
        and (data.loc[data["label_known"], "at_risk"] != data.loc[data["label_known"], "ready_by_start"]).all(),
}, name="passed")
assert checks.all(), "A data check failed."
checks.to_frame()

,passed
Cleaned request IDs are unique,True
Assessment IDs are unique,True
Every assessment matches one cleaned request,True
Assessment and request employee IDs agree,True
"at_risk is True, False, or blank only",True
"ready_by_start is True, False, or blank only",True
past_deadline_at_assessment is True or False only,True
Each task has one label across its checkpoints,True
No assessment after the extract,True
"Labels recalculate from timestamps (completed: finished after the deadline; unfinished: deadline passed = at risk, future deadline = unknown)",True


## B. Label eligibility

Assessments fall into three groups:

- **Early warning (eligible):** made before the deadline, with a known outcome. These are used for modelling, and the 90% recall target applies to them.
- **Overdue (reported separately):** made at or after the deadline, so the task is already late; always at risk.
- **Unknown outcome (not eligible):** the outcome was not known at the extract date. These are excluded and counted, never filled from the simulation.

The table gives each group's assessments, at-risk count, tasks, and employees, overall and by checkpoint.

In [3]:
data["group"] = np.select([~data["label_known"], data["past_deadline_at_assessment"] == "True"],
                          ["unknown outcome (not eligible)", "overdue (reported separately)"],
                          default="early warning (eligible)")
GROUPS = ["early warning (eligible)", "overdue (reported separately)", "unknown outcome (not eligible)"]
eligibility = (data.groupby(["group", "checkpoint"])
               .agg(assessments=("assessment_id", "size"), at_risk=("at_risk", lambda s: int((s == "True").sum())),
                    tasks=("request_id", "nunique"), employees=("employee_id", "nunique"))
               .reset_index())
totals = (data.groupby("group").agg(assessments=("assessment_id", "size"),
                                    at_risk=("at_risk", lambda s: int((s == "True").sum())),
                                    tasks=("request_id", "nunique"), employees=("employee_id", "nunique"))
          .reset_index().assign(checkpoint="all"))
eligibility = pd.concat([eligibility, totals])
eligibility["group"] = pd.Categorical(eligibility["group"], GROUPS, ordered=True)
eligibility = eligibility.sort_values(["group", "checkpoint"]).reset_index(drop=True)
eligibility["at_risk_%"] = (eligibility["at_risk"] / eligibility["assessments"] * 100).round(1)
# Unknown outcomes have no at-risk count; whole-number counts stay whole numbers
unknown_rows = eligibility["group"] == "unknown outcome (not eligible)"
eligibility["at_risk"] = eligibility["at_risk"].astype("Int64").mask(unknown_rows)
eligibility["at_risk_%"] = eligibility["at_risk_%"].mask(unknown_rows)
eligibility.insert(0, "data_version", DATA_VERSION)
eligibility.to_csv(OUTPUT_DIR / "eligibility_summary.csv", index=False)
eligibility

,data_version,group,checkpoint,assessments,at_risk,tasks,employees,at_risk_%
0,2,early warning (eligible),all,5173,949,4128,1026,18.3
1,2,early warning (eligible),checkpoint_1,4128,726,4128,1026,17.6
2,2,early warning (eligible),checkpoint_2,1045,223,1045,480,21.3
3,2,overdue (reported separately),all,191,191,191,71,100.0
4,2,overdue (reported separately),checkpoint_1,191,191,191,71,100.0
5,2,unknown outcome (not eligible),all,49,<NA>,44,18,NaN
6,2,unknown outcome (not eligible),checkpoint_1,44,<NA>,44,18,NaN
7,2,unknown outcome (not eligible),checkpoint_2,5,<NA>,5,3,NaN


## C. Model inputs and assessment-time availability

The model may use only information available when each assessment is made. The table lists each input, its source, and when it becomes known. Rather than relying on the data dictionary alone, the checks test the timing directly: checkpoint 1 happens exactly when information is complete, checkpoint 2 happens later, every assessment is before team completion, the recorded status matches the latest status event at the assessment time, and days until start, business days in the current status, and team workload all recalculate exactly from the timestamps and status events, using only earlier events. Numeric inputs must parse as whole numbers, and no input may be missing or empty.

**Cleaning assumption.** Cleaning filled a small number of blank departments and employee types from the same employee's other records. The generator confirms these values existed at each request's information check, but the cleaner may recover them from records created later. For this main analysis, which looks back over the full extract, this is treated as an acceptable recording repair and stated as an assumption.

**Missing intake channels** cannot be recovered, so they are kept as an explicit `unknown` category representing missing source information. Any encoding is fitted inside the training pipeline in a later notebook.

In [4]:
# Missing intake channels cannot be recovered; they are kept as an explicit "unknown" category
missing_channel = data["intake_channel"] == ""
data["intake_channel"] = data["intake_channel"].replace("", "unknown")

features = pd.DataFrame([
    ("task_type", "categorical", "cleaned requests", "at submission"),
    ("hiring_department", "categorical", "cleaned requests", "confirmed at the information check (checkpoint 1 or earlier)"),
    ("employee_type", "categorical", "cleaned requests", "at submission"),
    ("intake_channel", "categorical", "cleaned requests", "at submission; 'unknown' when missing at source"),
    ("info_issue_at_intake", "categorical", "cleaned requests", "at submission"),
    ("current_status", "categorical", "assessments", "status at the assessment time"),
    ("business_days_until_start", "numeric", "assessments", "calculated at the assessment time"),
    ("business_days_in_current_status", "numeric", "assessments", "calculated at the assessment time"),
    ("team_open_workload", "numeric", "assessments", "calculated from events before the assessment time"),
], columns=["feature", "type", "source", "known"])
features.insert(0, "data_version", DATA_VERSION)

# Evidence that each input is available at the assessment time
cp1, cp2 = data["checkpoint"] == "checkpoint_1", data["checkpoint"] == "checkpoint_2"
events_sorted = events.sort_values(["request_id", "event_at", "event_id"])


def status_at(row):
    """Status after the latest event at or before the assessment time."""
    past = events_by_request[row.request_id]
    return past.loc[past["event_at"] <= row.assessed_at, "status_after"].iloc[-1]


events_by_request = {rid: g for rid, g in events_sorted.groupby("request_id")}
team_of = requests.set_index("request_id")["responsible_team"]
assigned = events[events["event_type"] == "assigned_to_team"].set_index("request_id")["event_at"]
completed = events[events["event_type"] == "team_completed"].set_index("request_id")["event_at"]
team_tasks = pd.DataFrame({"team": team_of, "assigned": assigned, "completed": completed}).dropna(subset=["assigned"])
team_tasks["completed"] = team_tasks["completed"].fillna(pd.Timestamp.max)   # not yet completed at the extract


def workload_from_events(row):
    """Other tasks for the same team assigned strictly before the assessment and completing strictly after it."""
    t = team_tasks[(team_tasks["team"] == row.responsible_team) & (team_tasks.index != row.request_id)]
    return int(((t["assigned"] < row.assessed_at) & (t["completed"] > row.assessed_at)).sum())


days_until = [generator.business_days_until_start(a.date(), s.date())
              for a, s in zip(data["assessed_at"], data["confirmed_start_date"])]
# Status start: information completion while open, assignment once in progress (documented convention)
status_since = data["info_complete_at"].where(data["current_status"] == "open", data["assigned_at"])
days_in_status = [generator.business_days_between(since.date(), a.date())
                  for since, a in zip(status_since, data["assessed_at"])]
NUMERIC = features.loc[features["type"] == "numeric", "feature"].tolist()
numeric_values = data[NUMERIC].apply(pd.to_numeric, errors="coerce")
model_values = data[features["feature"]]
availability = pd.Series({
    "Checkpoint 1 happens exactly when information is complete": (data.loc[cp1, "assessed_at"] == data.loc[cp1, "info_complete_at"]).all(),
    "Checkpoint 2 happens after information is complete": (data.loc[cp2, "assessed_at"] > data.loc[cp2, "info_complete_at"]).all(),
    "Every assessment happens before the task is team-completed":
        (data["team_completed_at"].isna() | (data["assessed_at"] < data["team_completed_at"])).all(),
    "Current status equals the latest event status at the assessment time":
        (data.apply(status_at, axis=1) == data["current_status"]).all(),
    "Days until start recalculates with the closure calendar": (pd.Series(days_until, index=data.index)
                                                                == numeric_values["business_days_until_start"]).all(),
    "Business days in current status recalculates from the status start (closure calendar)":
        (pd.Series(days_in_status, index=data.index) == numeric_values["business_days_in_current_status"]).all(),
    "Team workload recalculates from status events (no future information)":
        (data.apply(workload_from_events, axis=1) == numeric_values["team_open_workload"]).all(),
    "No input is an outcome, label, simulation, or answer-key field":
        not set(features["feature"]) & {"team_completed_at", "completion_confirmed_at", "ready_by_start", "at_risk",
                                        "past_deadline_at_assessment", "status_at_extract"},
    "Numeric inputs parse as whole numbers": numeric_values.notna().all().all() and (numeric_values % 1 == 0).all().all(),
    "Time in status and team workload are never negative":
        (numeric_values[["business_days_in_current_status", "team_open_workload"]] >= 0).all().all(),
    "No missing or empty inputs after marking missing channels as 'unknown'":
        model_values.notna().all().all() and model_values.ne("").all().all(),
}, name="passed")
assert availability.all(), "An input-availability check failed."
features.to_csv(OUTPUT_DIR / "feature_list.csv", index=False)
print(f"Assessments with intake channel 'unknown' (missing at source): {int(missing_channel.sum())} "
      f"from {data.loc[missing_channel, 'request_id'].nunique()} requests")
display(availability.to_frame())
features

Assessments with intake channel 'unknown' (missing at source): 56 from 44 requests


,passed
Checkpoint 1 happens exactly when information is complete,True
Checkpoint 2 happens after information is complete,True
Every assessment happens before the task is team-completed,True
Current status equals the latest event status at the assessment time,True
Days until start recalculates with the closure calendar,True
Business days in current status recalculates from the status start (closure calendar),True
Team workload recalculates from status events (no future information),True
"No input is an outcome, label, simulation, or answer-key field",True
Numeric inputs parse as whole numbers,True
Time in status and team workload are never negative,True


,data_version,feature,type,source,known
0,2,task_type,categorical,cleaned requests,at submission
1,2,hiring_department,categorical,cleaned requests,confirmed at the information check (checkpoint...
2,2,employee_type,categorical,cleaned requests,at submission
3,2,intake_channel,categorical,cleaned requests,at submission; 'unknown' when missing at source
4,2,info_issue_at_intake,categorical,cleaned requests,at submission
5,2,current_status,categorical,assessments,status at the assessment time
6,2,business_days_until_start,numeric,assessments,calculated at the assessment time
7,2,business_days_in_current_status,numeric,assessments,calculated at the assessment time
8,2,team_open_workload,numeric,assessments,calculated from events before the assessment time


## D. Employee split

All of one employee's tasks and assessments stay in the same split, so no employee contributes to both model development and evaluation. Employees with assessments are divided approximately 60/20/20 into training, validation, and test, with seed 2026, using the same method as AIDA 1141 Exercise 05.

The split is stratified by employee type and the number of the employee's tasks known to have missed the deadline (0, 1, or 2+). Employee types are first checked to be allowed values with one type per employee. Late tasks are counted once per request, not once per assessment, and tasks with an unknown outcome are not counted as late. This outcome information is used only to balance the splits; it is never a model input.

The split is saved and protected: a rerun must reproduce it exactly, or the notebook stops. This is the main analysis, which measures performance on held-out employees within the simulated period.

In [5]:
# One row per employee; late tasks are counted once per request from the requests' readiness
# (an outcome, used only to balance the split; never a model input). Unknown outcomes are not counted as late.
emp_requests = requests.groupby("employee_id")
EMPLOYEE_TYPES = {"permanent", "temporary", "seasonal"}
assert requests["employee_type"].isin(EMPLOYEE_TYPES).all(), "An employee type is blank or not an allowed value."
assert emp_requests["employee_type"].nunique().eq(1).all(), "An employee has more than one employee type."
employees = pd.DataFrame({
    "employee_type": emp_requests["employee_type"].first(),
    "tasks": emp_requests.size(),
    "late_tasks": emp_requests["ready_by_start"].agg(lambda s: int((s == "False").sum())),
    "unknown_outcome_tasks": emp_requests["ready_by_start"].agg(lambda s: int((s == "").sum())),
}).reset_index()
employees = employees[employees["employee_id"].isin(data["employee_id"])]      # employees with assessments
employees["late_group"] = pd.cut(employees["late_tasks"], [-1, 0, 1, np.inf], labels=["0", "1", "2+"]).astype(str)

rng = np.random.default_rng(SEED)
parts = []
for _, group in employees.groupby(["employee_type", "late_group"]):
    ids = group["employee_id"].to_numpy().copy()
    rng.shuffle(ids)
    position = np.arange(len(ids))
    labels = np.where(position < round(0.6 * len(ids)), "train",
                      np.where(position < round(0.8 * len(ids)), "validation", "test"))
    parts.append(pd.DataFrame({"employee_id": ids, "split": labels}))
employee_split = pd.concat(parts).sort_values("employee_id").reset_index(drop=True)
employee_split.insert(0, "data_version", DATA_VERSION)

# Later notebooks rely on this file, so a rerun must reproduce it exactly rather than silently replace it
split_file = OUTPUT_DIR / "employee_split.csv"
if split_file.exists():
    saved = pd.read_csv(split_file, dtype=str)
    assert saved.equals(employee_split.astype(str)), "The rebuilt split differs from the saved employee_split.csv."
    print("Split matches the saved employee_split.csv")
else:
    employee_split.to_csv(split_file, index=False)
    print("Saved a new employee_split.csv")

data = data.merge(employee_split[["employee_id", "split"]], on="employee_id", how="left", validate="many_to_one")
split_checks = pd.Series({
    "Every employee with assessments is assigned exactly once":
        employee_split["employee_id"].is_unique and len(employee_split) == data["employee_id"].nunique(),
    "Every assessment has a split": data["split"].notna().all(),
    "No task appears in more than one split": data.groupby("request_id")["split"].nunique().max() == 1,
}, name="passed")
assert split_checks.all(), "The split is inconsistent."
SPLITS = ["train", "validation", "test"]
display(split_checks.to_frame())
(employees.merge(employee_split, on="employee_id")
 .pivot_table(index=["employee_type", "late_group"], columns="split", values="employee_id", aggfunc="size", fill_value=0)[SPLITS])

Split matches the saved employee_split.csv


,passed
Every employee with assessments is assigned exactly once,True
Every assessment has a split,True
No task appears in more than one split,True


split                     train  validation  test
employee_type late_group                         
permanent     0             275          91    92
              1              35          11    12
              2+             34          11    11
seasonal      0              60          20    20
              1              35          12    12
              2+             69          23    23
temporary     0              65          22    22
              1              25           8     8
              2+             46          16    15

## E. Split summary

For each split: employees, tasks with at least one assessment (requests still awaiting their first assessment are not included), eligible early-warning assessments and how many are at risk, overdue assessments, and assessments with an unknown outcome. Stratifying reduces differences in the at-risk share between splits but does not guarantee identical groups.

Every check in this notebook is also saved, with its section, to `outputs/checks.csv`.

In [6]:
early = data[data["group"] == "early warning (eligible)"]
split_summary = pd.DataFrame({
    "employees": employee_split["split"].value_counts(),
    "tasks_with_assessments": data.groupby("split")["request_id"].nunique(),
    "early_warning_assessments": early.groupby("split").size(),
    "early_warning_at_risk": early.groupby("split")["at_risk"].agg(lambda s: int((s == "True").sum())),
    "overdue_assessments": data[data["group"] == "overdue (reported separately)"].groupby("split").size(),
    "unknown_outcome_assessments": data[data["group"] == "unknown outcome (not eligible)"].groupby("split").size(),
}).reindex(SPLITS).fillna(0).astype(int)
split_summary["early_warning_at_risk_%"] = (split_summary["early_warning_at_risk"]
                                            / split_summary["early_warning_assessments"] * 100).round(1)
split_summary.insert(0, "data_version", DATA_VERSION)
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv")
gap = split_summary["early_warning_at_risk_%"].max() - split_summary["early_warning_at_risk_%"].min()
# Every check from this notebook, saved with its section
pd.concat({"A. data and labels": checks, "C. input availability": availability, "D. split": split_checks},
          names=["section", "check"]).rename("passed").to_csv(OUTPUT_DIR / "checks.csv")
print(f"Largest gap in early-warning at-risk share between splits: {gap:.1f} percentage points")
split_summary

Largest gap in early-warning at-risk share between splits: 0.6 percentage points


,data_version,employees,tasks_with_assessments,early_warning_assessments,early_warning_at_risk,overdue_assessments,unknown_outcome_assessments,early_warning_at_risk_%
split,,,,,,,,
train,2,644,2615,3084,563,126,34,18.3
validation,2,214,871,1033,194,24,8,18.8
test,2,215,877,1056,192,41,7,18.2


## F. Time-based check (defined before any results)

A secondary check will measure performance on a later period, which the employee split cannot show because its splits cover overlapping dates. Its dates and procedure are fixed here, before any model is trained or any result is seen, and saved to `outputs/time_check_definition.json`. Like the split, the saved definition is protected: a rerun must reproduce it exactly, or the notebook stops.

- **Training** uses only outcomes already known by the cutoff. Dividing employees by start date alone is not enough, because a task assessed before the cutoff may only finish afterwards.
- **Evaluation** uses assessments on or after the cutoff from employees with no earlier assessments. Employees spanning the cutoff keep their eligible pre-cutoff records in training, and their post-cutoff assessments are excluded from evaluation.
- **Isolation:** the model and threshold procedures for this check and for the main split are decided and frozen before either held-out result is inspected, so neither experiment can inform the other.
- **Limitation:** the cleaned requests were made retrospectively from the full extract, so a few repaired values may come from records created after the cutoff.

The table reports group sizes only; no model results.

In [7]:
# Time-based check: defined here, before any model is trained or any result is seen.
CUTOFF = pd.Timestamp("2026-01-01 00:00")
time_check = {
    "data_version": DATA_VERSION,
    "purpose": "Secondary check of performance on a later period; the employee split above remains the main analysis.",
    "cutoff": str(CUTOFF),
    "training_rule": ("Early-warning assessments made before the cutoff whose outcome was already known at the cutoff: "
                      "the team finished by the cutoff, or the deadline had passed by the cutoff while the task was "
                      "unfinished. Outcomes that became known only after the cutoff are not used for training."),
    "evaluation_rule": ("Early-warning assessments made on or after the cutoff, with an outcome known at the extract, "
                        "from employees with no assessment before the cutoff."),
    "threshold_rule": "Chosen within the training period only (grouped by employee), never on the evaluation period.",
    "measures": "The same measures as the main analysis: recall (primary), precision, flags, and unique tasks flagged.",
    "seasons": "Both periods include a spring peak (2025 in training, 2026 in evaluation); differences are reported as deployment findings.",
    "spanning_employees": ("Employees with assessments on both sides of the cutoff keep their eligible pre-cutoff records in "
                           "training; their post-cutoff assessments are excluded from evaluation."),
    "cleaning_limitation": ("The cleaned requests were made retrospectively from the full export, so a few repaired "
                            "values may come from records created after the cutoff; this limits the check's realism."),
    "isolation": ("The model and threshold procedures for this check and for the main employee split are decided and "
                  "frozen before either held-out result (main test set or evaluation period) is inspected."),
}
outcome_known_at_cutoff = (data["team_completed_at"] <= CUTOFF) | (data["deadline"] <= CUTOFF)
first_assessment = data.groupby("employee_id")["assessed_at"].transform("min")
period_train = early[(early["assessed_at"] < CUTOFF) & outcome_known_at_cutoff.loc[early.index]]
period_eval = early[(early["assessed_at"] >= CUTOFF) & (first_assessment.loc[early.index] >= CUTOFF)]
time_check["sizes"] = {  # sizes only; no model results
    "training_assessments": len(period_train), "training_employees": int(period_train["employee_id"].nunique()),
    "excluded_before_cutoff_outcome_not_yet_known": int(((early["assessed_at"] < CUTOFF)
                                                         & ~outcome_known_at_cutoff.loc[early.index]).sum()),
    "evaluation_assessments": len(period_eval), "evaluation_employees": int(period_eval["employee_id"].nunique()),
    "excluded_employees_spanning_the_cutoff": int(data.loc[(first_assessment < CUTOFF)
                                                          & (data["assessed_at"] >= CUTOFF), "employee_id"].nunique()),
}
# Like the split, the definition is protected: a rerun must reproduce it exactly rather than silently change it
definition_file = OUTPUT_DIR / "time_check_definition.json"
if definition_file.exists():
    assert json.loads(definition_file.read_text()) == time_check, \
        "The time-based check definition differs from the saved file; it must not change after it is fixed."
    print("Time-based check definition matches the saved file")
else:
    definition_file.write_text(json.dumps(time_check, indent=2))
    print("Saved the time-based check definition")
pd.Series(time_check["sizes"], name="count").to_frame()


Time-based check definition matches the saved file


,count
training_assessments,2683
training_employees,540
excluded_before_cutoff_outcome_not_yet_known,51
evaluation_assessments,2410
evaluation_employees,477
excluded_employees_spanning_the_cutoff,14
